## Model Selection Demo:

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

In [ ]:
# Simulate data
np.random.seed(0)
n = 1000
p = 5
X = np.random.randn(n, p) #after this assume X deterministic
beta = np.array([3, 2, 0, 0, 0])  # Only two variables are nonzero
y = X @ beta + np.random.randn(n) * 3 #y is still random bc of epsilon term

# Add a constant to X for intercept
X = sm.add_constant(X)

In [ ]:
# plot the data
import matplotlib.pyplot as plt
plt.scatter(X[:, 1], y, label='X1 vs y')

In [ ]:
# Best subset selection
def best_subset_selection(X, y):
    n, p = X.shape
    models = []
    
    for k in range(1, p + 1):  # Iterate over subset sizes
        for combo in combinations(range(1, p), k):  # Generate combinations of predictors
            combo = (0,) + combo  # Include the intercept
            X_subset = X[:, combo]
            model = sm.OLS(y, X_subset).fit()
            models.append((model, combo))
    
    return models


In [ ]:
print(list(combinations(range(1, 5), 2)))

In [ ]:
combo = (0,) + combinations(range(1, 5), 2).__next__()  # Get the first combination of size 2
combo

In [ ]:
full_model = sm.OLS(y, X).fit()

In [ ]:

# Calculate metrics
def calculate_metrics(model, X, y, full_model):
    n = len(y)
    k = model.df_model  # Number of predictors, excluding intercept


    
    # AIC
    aic = model.aic
    
    # BIC
    bic = model.bic
    
    
    # PRESS (Prediction Sum of Squares)
    hat_matrix = X @ np.linalg.inv(X.T @ X) @ X.T #X(XTX)^-1XT
    residuals = model.resid
    press = np.sum((residuals / (1 - np.diag(hat_matrix))) ** 2) #diag pulls out the leverage
    
    # Adjusted R-squared
    r2 = model.rsquared
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)

    #Mallow's Cp
    sse_k = (np.sum(model.resid**2))
    mse_full = sum((full_model.resid)**2) / (n - p)
    cp =  sse_k / mse_full + 2*(k + 1) - n
    
    return aic, bic, press, adj_r2, cp, int(k) #dont consider intercept as a predictor



In [ ]:
# Run best subset selection
models = best_subset_selection(X, y)

In [ ]:
# Store results in pd DataFrame
results = []
for model, combo in models:
    aic, bic, press, adj_r2, cp, num_predictors = calculate_metrics(model, X[:, combo], y, full_model)
    results.append({
        'Predictors': combo,
        'n_Predictors': num_predictors,
        'AIC': aic,
        'BIC': bic,
        'PRESS': press,
        'Adjusted R^2': adj_r2,
        'Mallows C_p': cp
    })

# Convert results to pd DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='n_Predictors').reset_index(drop=True)


In [ ]:
# Display our results
pd.set_option('display.max_columns', None)  # Show all columns
results_df #0 represents intercept – all models include intercept